# Corrective RAG (CRAG) Implementation

## Introduction

In this notebook, I am implementing a simple **Corrective RAG (CRAG)** system.

The main idea of Corrective RAG is to check the quality of the retrieved information before using it to generate an answer. Instead of blindly trusting the first search results, the system evaluates them and takes corrective action when the retrieved information is not good enough.

If the retrieved information is useful, the system directly generates the answer. If the information is unclear or irrelevant, the system rewrites the question and performs another search.

### Main steps in this notebook

1. Create a small knowledge base
2. Split documents into smaller chunks
3. Generate embeddings using Sentence Transformers
4. Store embeddings in FAISS
5. Perform semantic search using FAISS
6. Perform keyword search using BM25
7. Combine both searches using Reciprocal Rank Fusion (RRF)
8. Evaluate the retrieved information
9. Classify the retrieval as GOOD, AMBIGUOUS, or BAD
10. Rewrite the query when the retrieval is not good enough
11. Perform corrective retrieval
12. Generate the final answer using the retrieved evidence
13. Evaluate whether the final answer is supported
14. Display the final answer and retrieval process

### Architecture

```text
                    User Question
                          |
                          v
                   Initial Search
                          |
                  +-------+-------+
                  |               |
                FAISS            BM25
                  |               |
                  +-------+-------+
                          |
                          v
                   RRF Hybrid Search
                          |
                          v
                 Retrieval Evaluation
                          |
             +------------+------------+
             |            |            |
             v            v            v
           GOOD       AMBIGUOUS       BAD
             |            |            |
             |            v            v
             |       Rewrite Query   Rewrite Query
             |            |            |
             |            +-----+------+
             |                  |
             |                  v
             |          Corrective Search
             |                  |
             +------------------+
                        |
                        v
                 Retrieved Evidence
                        |
                        v
                 Answer Generation
                        |
                        v
                  Answer Evaluation
                        |
                        v
                    Final Answer

In [51]:
!pip -q install -U sentence-transformers faiss-cpu google-genai rank-bm25

In [52]:
import os
import re
import json
import numpy as np
import faiss

from getpass import getpass
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from google import genai

print("Libraries imported successfully.")

Libraries imported successfully.


In [53]:
GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

client = genai.Client(
    api_key=GEMINI_API_KEY
)

MODEL_NAME = "gemini-3.5-flash-lite"

print("Gemini initialized successfully.")

Enter your Gemini API key: ··········
Gemini initialized successfully.


In [54]:
documents = [

    {
        "id": "doc1",
        "title": "RAG Introduction",
        "text": """
        Retrieval-Augmented Generation, commonly called RAG, combines
        information retrieval with large language models. RAG retrieves
        relevant information from an external knowledge base and provides
        that information to the language model as context.
        """
    },

    {
        "id": "doc2",
        "title": "Vector Databases",
        "text": """
        A vector database stores numerical representations of data called
        embeddings. These embeddings are used for semantic similarity search.
        A question can be converted into an embedding and compared with
        document embeddings.
        """
    },

    {
        "id": "doc3",
        "title": "Embeddings",
        "text": """
        Embeddings are numerical vectors that represent the semantic meaning
        of text. Similar pieces of text tend to have embeddings that are close
        to each other in vector space. Embeddings are commonly used for
        semantic search and RAG.
        """
    },

    {
        "id": "doc4",
        "title": "RAG Pipeline",
        "text": """
        A typical RAG pipeline consists of document ingestion, text splitting,
        embedding generation, vector storage, retrieval, context construction,
        and language model generation.
        """
    },

    {
        "id": "doc5",
        "title": "Document Chunking",
        "text": """
        Chunking divides large documents into smaller pieces before embedding.
        Good chunking is important because large chunks can contain irrelevant
        information while very small chunks may lose important context.
        """
    },

    {
        "id": "doc6",
        "title": "Semantic Search",
        "text": """
        Semantic search retrieves information based on meaning rather than
        only exact keyword matching. Queries and documents are converted into
        embeddings and their similarity is calculated.
        """
    },

    {
        "id": "doc7",
        "title": "BM25 Keyword Search",
        "text": """
        BM25 is a keyword-based information retrieval algorithm. It is useful
        for exact keywords, technical terminology, identifiers and error codes.
        """
    },

    {
        "id": "doc8",
        "title": "Hybrid Search",
        "text": """
        Hybrid search combines semantic vector search and lexical keyword
        search. Vector search is useful for meaning while keyword search is
        useful for exact terms.
        """
    },

    {
        "id": "doc9",
        "title": "Authentication Errors",
        "text": """
        Authentication errors can occur when credentials are invalid,
        authentication tokens expire, or authorization policies reject a request.

        ERR-401 indicates an authentication failure.

        ERR-403 indicates that the user is authenticated but does not have
        permission to access a resource.

        Access tokens authenticate requests. Refresh tokens can be used to
        obtain new access tokens when an access token expires.
        """
    },

    {
        "id": "doc10",
        "title": "Product API",
        "text": """
        The Product API provides endpoints for creating, updating, deleting
        and retrieving product records. Product records contain a product ID,
        name, price, inventory quantity and category. The API uses JSON.
        """
    },

    {
        "id": "doc11",
        "title": "Query Rewriting",
        "text": """
        Query rewriting transforms a user's original question into a clearer
        and more retrieval-friendly query. It can make important concepts
        explicit and use terminology that better matches the knowledge base.
        """
    },

    {
        "id": "doc12",
        "title": "Query Decomposition",
        "text": """
        Query decomposition breaks a complex question into smaller
        sub-questions. Each sub-question can be answered independently using
        retrieval and the results can then be combined.
        """
    },

    {
        "id": "doc13",
        "title": "Contextual Retrieval",
        "text": """
        Contextual retrieval improves document chunks by adding information
        about where each chunk came from and what it means in the broader
        document.
        """
    },

    {
        "id": "doc14",
        "title": "Reranker",
        "text": """
        A reranker takes a query and candidate documents and calculates
        relevance scores. It is commonly used after an initial retrieval step.
        """
    },

    {
        "id": "doc15",
        "title": "Corrective RAG",
        "text": """
        Corrective RAG, or CRAG, evaluates the quality of retrieved documents
        before using them. If retrieval is good, the system generates an
        answer. If retrieval is ambiguous, the system can rewrite the query
        and search again. If retrieval is poor, the system can use an
        alternative retrieval strategy.
        """
    }
]

print("Number of documents:", len(documents))

Number of documents: 15


In [55]:
def chunk_text(text, chunk_size=70, overlap=15):

    words = text.split()
    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        if chunk.strip():
            chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap

    return chunks

In [56]:
chunks = []

for document in documents:

    document_chunks = chunk_text(
        document["text"]
    )

    for number, chunk in enumerate(document_chunks):

        chunks.append({
            "chunk_id": f'{document["id"]}_chunk_{number}',
            "document_id": document["id"],
            "title": document["title"],
            "chunk_number": number,
            "text": chunk
        })

print("Total chunks:", len(chunks))

Total chunks: 15


In [57]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embedding_dimension = (
    embedding_model.get_sentence_embedding_dimension()
)

print("Embedding dimension:", embedding_dimension)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimension: 384


/tmp/ipykernel_5093/2176670758.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension()


In [58]:
chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

print("Embedding shape:", embeddings.shape)

Embedding shape: (15, 384)


In [59]:
vector_index = faiss.IndexFlatIP(
    embedding_dimension
)

vector_index.add(embeddings)

print("FAISS vectors:", vector_index.ntotal)

FAISS vectors: 15


In [60]:
def tokenize(text):

    return re.findall(
        r"\b\w+\b",
        text.lower()
    )


tokenized_chunks = [
    tokenize(chunk["text"])
    for chunk in chunks
]

bm25 = BM25Okapi(
    tokenized_chunks
)

print("BM25 index created.")

BM25 index created.


In [61]:
def vector_search(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = vector_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, (score, index) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        result = chunks[index].copy()

        result["vector_score"] = float(score)
        result["vector_rank"] = rank

        results.append(result)

    return results

In [62]:
def bm25_search(query, top_k=5):

    scores = bm25.get_scores(
        tokenize(query)
    )

    indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, index in enumerate(
        indices,
        start=1
    ):

        result = chunks[index].copy()

        result["bm25_score"] = float(
            scores[index]
        )

        result["bm25_rank"] = rank

        results.append(result)

    return results

In [63]:
def reciprocal_rank_fusion(
    result_lists,
    k=60
):

    fused = {}

    for results in result_lists:

        for rank, result in enumerate(
            results,
            start=1
        ):

            chunk_id = result["chunk_id"]

            if chunk_id not in fused:

                fused[chunk_id] = {
                    "chunk": result,
                    "score": 0.0
                }

            fused[chunk_id]["score"] += (
                1 / (k + rank)
            )

    ranked = sorted(
        fused.values(),
        key=lambda x: x["score"],
        reverse=True
    )

    final_results = []

    for item in ranked:

        result = item["chunk"].copy()

        result["rrf_score"] = item["score"]

        final_results.append(result)

    return final_results

In [64]:
def hybrid_search(query, top_k=5):

    vector_results = vector_search(
        query,
        top_k=10
    )

    bm25_results = bm25_search(
        query,
        top_k=10
    )

    fused_results = reciprocal_rank_fusion(
        [
            vector_results,
            bm25_results
        ]
    )

    return fused_results[:top_k]

In [65]:
question = "What does ERR-401 mean?"

results = hybrid_search(
    question,
    top_k=5
)

for i, result in enumerate(
    results,
    start=1
):

    print("-" * 60)
    print("Rank:", i)
    print("Title:", result["title"])
    print("RRF Score:", round(
        result["rrf_score"],
        4
    ))
    print(result["text"])

------------------------------------------------------------
Rank: 1
Title: Authentication Errors
RRF Score: 0.0328
Authentication errors can occur when credentials are invalid, authentication tokens expire, or authorization policies reject a request. ERR-401 indicates an authentication failure. ERR-403 indicates that the user is authenticated but does not have permission to access a resource. Access tokens authenticate requests. Refresh tokens can be used to obtain new access tokens when an access token expires.
------------------------------------------------------------
Rank: 2
Title: Corrective RAG
RRF Score: 0.0317
Corrective RAG, or CRAG, evaluates the quality of retrieved documents before using them. If retrieval is good, the system generates an answer. If retrieval is ambiguous, the system can rewrite the query and search again. If retrieval is poor, the system can use an alternative retrieval strategy.
------------------------------------------------------------
Rank: 3
Title:

In [66]:
def build_context(results):

    context = ""

    for i, result in enumerate(
        results,
        start=1
    ):

        context += f"""
SOURCE {i}
TITLE: {result["title"]}

{result["text"]}

"""

    return context

In [67]:
def evaluate_retrieval(
    question,
    results
):

    context = build_context(results)

    prompt = f"""
You are a Corrective RAG retrieval evaluator.

Check whether the retrieved information is useful
for answering the question.

Return ONLY JSON:

{{
    "grade": "GOOD",
    "score": 1.0,
    "reason": "short explanation"
}}

Use:

GOOD = directly useful
AMBIGUOUS = partly useful or missing information
BAD = mostly irrelevant

QUESTION:
{question}

RETRIEVED INFORMATION:
{context}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    text = response.text.strip()

    text = text.replace(
        "```json",
        ""
    ).replace(
        "```",
        ""
    ).strip()

    try:

        result = json.loads(text)

        grade = result.get(
            "grade",
            "AMBIGUOUS"
        ).upper()

        if grade not in [
            "GOOD",
            "AMBIGUOUS",
            "BAD"
        ]:
            grade = "AMBIGUOUS"

        score = float(
            result.get("score", 0.0)
        )

        return {
            "grade": grade,
            "score": max(
                0.0,
                min(1.0, score)
            ),
            "reason": result.get(
                "reason",
                ""
            )
        }

    except Exception:

        return {
            "grade": "AMBIGUOUS",
            "score": 0.4,
            "reason": "Could not evaluate retrieval."
        }

In [68]:
def rewrite_query(
    question,
    feedback=""
):

    prompt = f"""
Rewrite the question so that it is easier
to search in the knowledge base.

Keep the original meaning.

Return ONLY JSON:

{{
    "rewritten_query": "...",
    "reason": "short explanation"
}}

QUESTION:
{question}

RETRIEVAL FEEDBACK:
{feedback}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    text = response.text.strip()

    text = text.replace(
        "```json",
        ""
    ).replace(
        "```",
        ""
    ).strip()

    try:

        result = json.loads(text)

        return {
            "rewritten_query": result.get(
                "rewritten_query",
                question
            ),
            "reason": result.get(
                "reason",
                ""
            )
        }

    except Exception:

        return {
            "rewritten_query": question,
            "reason": "Original query retained."
        }

In [69]:
def alternative_retrieval(
    query,
    top_k=5
):

    vector_results = vector_search(
        query,
        top_k=10
    )

    bm25_results = bm25_search(
        query,
        top_k=10
    )

    results = reciprocal_rank_fusion(
        [
            vector_results,
            bm25_results
        ],
        k=30
    )

    return results[:top_k]

In [70]:
def generate_answer(
    question,
    context
):

    prompt = f"""
You are a Corrective RAG answer generator.

Answer the question using the evidence.

Rules:
1. Use the evidence as the main source.
2. Do not invent facts.
3. If the evidence is insufficient, say so.
4. Give a clear answer.

QUESTION:
{question}

EVIDENCE:
{context}

ANSWER:
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    return response.text.strip()

In [71]:
def evaluate_answer(
    question,
    answer,
    context
):

    prompt = f"""
Check whether the answer is supported by
the provided evidence.

Return ONLY JSON:

{{
    "supported": true,
    "score": 1.0,
    "reason": "short explanation"
}}

QUESTION:
{question}

EVIDENCE:
{context}

ANSWER:
{answer}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    text = response.text.strip()

    text = text.replace(
        "```json",
        ""
    ).replace(
        "```",
        ""
    ).strip()

    try:

        result = json.loads(text)

        score = float(
            result.get("score", 0.0)
        )

        return {
            "supported": bool(
                result.get(
                    "supported",
                    score >= 0.7
                )
            ),
            "score": max(
                0.0,
                min(1.0, score)
            ),
            "reason": result.get(
                "reason",
                ""
            )
        }

    except Exception:

        return {
            "supported": False,
            "score": 0.0,
            "reason": "Answer evaluation failed."
        }

In [72]:
def corrective_rag(
    question,
    top_k=5,
    max_corrections=1
):

    current_query = question

    best_results = []
    best_score = 0

    trace = []

    for attempt in range(
        max_corrections + 1
    ):

        print(
            f"\nRetrieval attempt {attempt + 1}"
        )

        # Initial search
        if attempt == 0:

            results = hybrid_search(
                current_query,
                top_k
            )

        # Corrective search
        else:

            results = alternative_retrieval(
                current_query,
                top_k
            )

        context = build_context(
            results
        )

        # Evaluate retrieval
        evaluation = evaluate_retrieval(
            question,
            results
        )

        print(
            "Retrieval grade:",
            evaluation["grade"]
        )

        print(
            "Retrieval score:",
            evaluation["score"]
        )

        trace.append({
            "attempt": attempt + 1,
            "query": current_query,
            "evaluation": evaluation
        })

        # Save best result
        if evaluation["score"] > best_score:

            best_score = evaluation["score"]

            best_results = results

        # Good retrieval
        if evaluation["grade"] == "GOOD":

            print("Good evidence found.")
            break

        # No more corrections
        if attempt == max_corrections:

            print(
                "Maximum corrections reached."
            )
            break

        # Rewrite query
        rewrite = rewrite_query(
            question,
            evaluation["reason"]
        )

        current_query = rewrite[
            "rewritten_query"
        ]

        print(
            "New query:",
            current_query
        )

    # Build final context
    best_context = build_context(
        best_results
    )

    # Generate answer
    answer = generate_answer(
        question,
        best_context
    )

    # Evaluate answer
    answer_evaluation = evaluate_answer(
        question,
        answer,
        best_context
    )

    return {
        "question": question,
        "answer": answer,
        "sources": best_results,
        "retrieval_score": best_score,
        "answer_score": answer_evaluation["score"],
        "answer_supported": answer_evaluation["supported"],
        "trace": trace
    }

In [73]:
question = "What does ERR-401 mean?"

results = hybrid_search(
    question,
    top_k=5
)

evaluation = evaluate_retrieval(
    question,
    results
)

print("Question:", question)
print("Grade:", evaluation["grade"])
print("Score:", evaluation["score"])
print("Reason:", evaluation["reason"])

Question: What does ERR-401 mean?
Grade: GOOD
Score: 1.0
Reason: Source 1 directly explains that ERR-401 indicates an authentication failure.


In [74]:
question = "What does that login problem mean?"

results = hybrid_search(
    question,
    top_k=5
)

evaluation = evaluate_retrieval(
    question,
    results
)

rewrite = rewrite_query(
    question,
    evaluation["reason"]
)

print("Original question:")
print(question)

print("\nRetrieval grade:")
print(evaluation["grade"])

print("\nRetrieval reason:")
print(evaluation["reason"])

print("\nRewritten query:")
print(rewrite["rewritten_query"])

Original question:
What does that login problem mean?

Retrieval grade:
BAD

Retrieval reason:
The question is extremely vague ('that login problem') and the retrieved sources do not provide specific context for a particular login problem, other than general authentication error codes.

Rewritten query:
common authentication error codes and login failure troubleshooting


In [75]:
question = "What does ERR-401 mean and why can it occur?"

result = corrective_rag(
    question,
    top_k=5,
    max_corrections=1
)

print("\n" + "=" * 60)
print("FINAL CRAG RESULT")
print("=" * 60)

print("\nQuestion:")
print(result["question"])

print("\nAnswer:")
print(result["answer"])

print("\nRetrieval Score:")
print(
    round(
        result["retrieval_score"],
        3
    )
)

print("\nAnswer Score:")
print(
    round(
        result["answer_score"],
        3
    )
)

print("\nAnswer Supported:")
print(
    result["answer_supported"]
)

print("\nSources:")

for source in result["sources"]:

    print(
        "-",
        source["title"]
    )


Retrieval attempt 1
Retrieval grade: GOOD
Retrieval score: 1.0
Good evidence found.

FINAL CRAG RESULT

Question:
What does ERR-401 mean and why can it occur?

Answer:
Based on the provided evidence:

* **What ERR-401 means:** ERR-401 indicates an authentication failure.
* **Why it can occur:** Authentication errors (such as ERR-401) can occur when credentials are invalid, authentication tokens expire, or authorization policies reject a request.

Retrieval Score:
1.0

Answer Score:
1.0

Answer Supported:
True

Sources:
- Authentication Errors
- BM25 Keyword Search
- Reranker
- Contextual Retrieval
- Corrective RAG
